In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "bigcode/starcoder2-3b"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

model = model.to(device)

print("Model loaded!")

In [ ]:
import os
import glob
import math
import torch
from datasets import load_dataset
from transformers import (
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel
)

# Ensure tokenizer pad token is properly set for batching
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("Set pad_token to eos_token:", tokenizer.eos_token)

In [ ]:
# Locate dataset directory relative to project root
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
TRAIN_PATH = os.path.join(BASE_DIR, "dataset", "train", "*.jsonl")
VALID_PATH = os.path.join(BASE_DIR, "dataset", "valid", "*.jsonl")
TEST_PATH = os.path.join(BASE_DIR, "dataset", "test", "*.jsonl")

train_files = sorted(glob.glob(TRAIN_PATH))
valid_files = sorted(glob.glob(VALID_PATH))
test_files = sorted(glob.glob(TEST_PATH))

print(f"Found {len(train_files)} train file(s), {len(valid_files)} valid file(s), {len(test_files)} test file(s).")

raw_datasets = load_dataset("json", data_files={
    "train": train_files,
    "validation": valid_files,
    "test": test_files
})

print("\nRaw Datasets:")
print(raw_datasets)

# Option for rapid prototyping: use a subset of data for fast training iteration
# Set USE_SUBSET = False to train on the complete dataset
USE_SUBSET = True
TRAIN_SAMPLES = 20000
EVAL_SAMPLES = 2000

if USE_SUBSET:
    train_dataset = raw_datasets["train"].shuffle(seed=42).select(range(min(TRAIN_SAMPLES, len(raw_datasets["train"]))))
    valid_dataset = raw_datasets["validation"].shuffle(seed=42).select(range(min(EVAL_SAMPLES, len(raw_datasets["validation"]))))
    test_dataset = raw_datasets["test"].shuffle(seed=42).select(range(min(EVAL_SAMPLES, len(raw_datasets["test"]))))
else:
    train_dataset = raw_datasets["train"]
    valid_dataset = raw_datasets["validation"]
    test_dataset = raw_datasets["test"]

print(f"\nActive Training set size: {len(train_dataset)}")
print(f"Active Validation set size: {len(valid_dataset)}")
print(f"Active Test set size: {len(test_dataset)}")

In [ ]:
MAX_LENGTH = 512  # Maximum sequence length to keep memory footprint bounded

def preprocess_function(examples):
    # Use original_string (full function body) or fallback to code field
    texts = [
        doc if (doc and len(doc.strip()) > 0) else code
        for doc, code in zip(examples["original_string"], examples["code"])
    ]
    
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_tensors=None
    )
    
    # DataCollatorForLanguageModeling will automatically pad input_ids
    # and create appropriately padded labels with -100 for Causal LM training.
    return tokenized

print("Tokenizing datasets...")
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train dataset"
)

tokenized_valid = valid_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=valid_dataset.column_names,
    desc="Tokenizing validation dataset"
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names,
    desc="Tokenizing test dataset"
)

print("Tokenization complete. Sample features:", tokenized_train.column_names)

In [ ]:
peft_config = LoraConfig(
    r=16,                         # Rank dimension
    lora_alpha=32,                # Scaling factor
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", "c_fc", "c_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Enable gradient checkpointing for VRAM efficiency
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

# Wrap base model with LoRA adapter
model = get_peft_model(model, peft_config)

print("Trainable parameter breakdown:")
model.print_trainable_parameters()

In [ ]:
OUTPUT_DIR = os.path.join(BASE_DIR, "checkpoints", "starcoder2-code-completion")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,  # Effective batch size = 16
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    dataloader_num_workers=0,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
)

print("Trainer initialized!")

In [ ]:
print("Starting fine-tuning...")
train_result = trainer.train()

print("\nFine-tuning completed successfully!")
print(f"Total training steps: {train_result.global_step}")
print(f"Final training loss: {train_result.training_loss:.4f}")

In [ ]:
SAVE_PATH = os.path.join(BASE_DIR, "saved_models", "starcoder2-finetuned")

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"Fine-tuned model adapters saved to: {SAVE_PATH}")

In [ ]:
print("Evaluating on test dataset...")
test_results = trainer.evaluate(eval_dataset=tokenized_test)

test_loss = test_results.get("eval_loss")
perplexity = math.exp(test_loss) if test_loss is not None else float("nan")

print(f"\n=== Test Evaluation Results ===")
print(f"Test Loss:  {test_loss:.4f}")
print(f"Perplexity: {perplexity:.4f}")

In [ ]:
def generate_code_completion(prompt_text, max_new_tokens=150, temperature=0.2):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True if temperature > 0 else False,
            temperature=temperature,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt_1 = '''def calculate_factorial(n):
    """Calculates the factorial of a non-negative integer n."""
'''

print("=== Input Prompt ===")
print(prompt_1)
print("\n=== Generated Completion ===")
output_code = generate_code_completion(prompt_1)
print(output_code)